In [1]:
# Downgrade PyTorch to version that supports P100 (compute 6.0)
!pip install torch==2.4.0 torchvision==0.19.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q
print('Done!')

# Verify
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('P100 supported:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 3.4 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 29.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 86.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 32.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 14.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.6 MB/s eta 0:00:0000:0100:01
     ━━━━━

In [2]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(
        torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU! Go to Settings and enable GPU P100')

CUDA: True
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB


In [3]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install transformers pyarrow tqdm -q
print('Done')

Done


In [4]:
import os

MODEL_PATH = '/kaggle/input/datasets/yogendra2005/blair-custom-model/blair-videogames-multiaspect'
VOICE_PATH = '/kaggle/input/datasets/yogendra2005/blair-voice-docs/user_voice_docs.parquet'

print('Model files:')
for f in os.listdir(MODEL_PATH):
    size = os.path.getsize(f'{MODEL_PATH}/{f}') / 1024 / 1024
    print(f'  {f}: {size:.1f} MB')

print(f'\nVoice docs exists: {os.path.exists(VOICE_PATH)}')
print('\nAll files ready!')

Model files:
  config.json: 0.0 MB
  tokenizer.json: 3.4 MB
  tokenizer_config.json: 0.0 MB
  model.safetensors: 1355.6 MB

Voice docs exists: True

All files ready!


In [5]:
# ── Cell 4: Load data and check columns ────────────────────────────────────
import pandas as pd

df = pd.read_parquet(VOICE_PATH)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('First row user_id:', df.iloc[0]['user_id'])
print('Sample voice doc:')
print(df.iloc[0]['voice_document'][:300])

Shape: (94762, 3)
Columns: ['user_id', 'voice_document', 'tier']
First row user_id: AE222HFZDH6BPTYFOUWGGU63YSIQ
Sample voice doc:
User taste profile:


=== INTERACTION HISTORY ===
Total interactions: 3
Active for: 172 days
Engagement intensity: 0.017 interactions/day
Rating pattern: avg 5.0/5 (std 0.00)
Critic style: generous

=== CATEGORY PREFERENCES ===
Favorite categories: Headsets, Games, Consoles
Taste diversity: 1.000
Ca


In [6]:
import torch
import subprocess

# Check CUDA version
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

# Check PyTorch CUDA
print('PyTorch version:', torch.__version__)
print('CUDA version PyTorch built with:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU Compute Capability:', torch.cuda.get_device_capability(0))

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

PyTorch version: 2.4.0+cu121
CUDA version PyTorch built with: 12.1
GPU: Tesla P100-PCIE-16GB
GPU Compute Capability: (6, 0)


In [7]:
# ── Cell 5: Generate user voice embeddings ─────────────────────────────────
import time
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from pathlib import Path

# Config
BATCH_SIZE       = 64
MAX_LENGTH       = 512
EMB_DIM          = 1024
OUT_EMB          = '/kaggle/working/user_voice_embeddings.npy'
OUT_IDS          = '/kaggle/working/user_voice_ids.npy'
PARTIAL_EMB      = '/kaggle/working/voice_partial.npy'
LAST_IDX_FILE    = '/kaggle/working/voice_last_idx.txt'

# Load data
# Try both possible column names
if 'voice_document' in df.columns:
    texts = df['voice_document'].fillna('').tolist()
elif 'voice_doc' in df.columns:
    texts = df['voice_doc'].fillna('').tolist()
elif 'rich_text' in df.columns:
    texts = df['rich_text'].fillna('').tolist()
else:
    # Use first text column found
    text_cols = [c for c in df.columns if df[c].dtype == object]
    print('Available text columns:', text_cols)
    texts = df[text_cols[0]].fillna('').tolist()
    print(f'Using column: {text_cols[0]}')

uids    = df['user_id'].tolist()
n_users = len(texts)
print(f'Loaded {n_users:,} users')
print(f'Sample text length: {len(texts[0])} chars')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load model
print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModel.from_pretrained(MODEL_PATH).to(device).eval()
print('Model ready!')

# Resume from checkpoint if exists
start_idx  = 0
embeddings = np.zeros((n_users, EMB_DIM), dtype=np.float32)
if Path(PARTIAL_EMB).exists() and Path(LAST_IDX_FILE).exists():
    start_idx = int(Path(LAST_IDX_FILE).read_text().strip())
    partial   = np.load(PARTIAL_EMB)
    if partial.shape == (start_idx, EMB_DIM):
        embeddings[:start_idx] = partial
        print(f'Resumed from checkpoint: {start_idx}/{n_users}')
    else:
        start_idx = 0
        print('Checkpoint mismatch — starting fresh')

# Embedding loop
t0 = time.time()
for batch_start in tqdm(
        range(start_idx, n_users, BATCH_SIZE), desc='Voice encoding'):

    batch_end   = min(batch_start + BATCH_SIZE, n_users)
    batch_texts = texts[batch_start:batch_end]

    enc = tokenizer(
        batch_texts,
        max_length=MAX_LENGTH,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)

    cls = out.last_hidden_state[:, 0, :]
    cls = F.normalize(cls, dim=-1)
    embeddings[batch_start:batch_end] = cls.cpu().float().numpy()

    # Save checkpoint every 5000 users
    if batch_end % 5000 < BATCH_SIZE:
        np.save(PARTIAL_EMB, embeddings[:batch_end])
        Path(LAST_IDX_FILE).write_text(str(batch_end))
        elapsed = time.time() - t0
        rate    = batch_end / elapsed
        eta     = (n_users - batch_end) / rate / 60
        print(f'Checkpoint: {batch_end}/{n_users} | '
              f'{rate:.0f} users/s | ETA {eta:.1f} min')

elapsed = time.time() - t0
print(f'\nDone: {n_users:,} users in {elapsed/60:.1f} min')

# Save final output
np.save(OUT_EMB, embeddings)
np.save(OUT_IDS, np.array(uids, dtype=object))
print(f'Saved user_voice_embeddings.npy shape={embeddings.shape}')
print(f'Saved user_voice_ids.npy count={len(uids)}')

# Cleanup checkpoints
for f in [PARTIAL_EMB, LAST_IDX_FILE]:
    if Path(f).exists(): Path(f).unlink()

Loaded 94,762 users
Sample text length: 1163 chars
Device: cuda
Loading model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model ready!


Voice encoding:   5%|▌         | 79/1481 [04:18<1:16:44,  3.28s/it]

Checkpoint: 5056/94762 | 20 users/s | ETA 76.3 min


...

Checkpoint: 85056/94762 | 20 users/s | ETA 8.3 min


Voice encoding:  95%|█████████▌| 1407/1481 [1:16:50<04:08,  3.35s/it]

Checkpoint: 90048/94762 | 20 users/s | ETA 4.0 min


Voice encoding: 100%|██████████| 1481/1481 [1:20:51<00:00,  3.28s/it]



Done: 94,762 users in 80.9 min
Saved user_voice_embeddings.npy shape=(94762, 1024)
Saved user_voice_ids.npy count=94762


In [8]:
# ── Cell 6: Verify outputs ─────────────────────────────────────────────────
import numpy as np

emb = np.load('/kaggle/working/user_voice_embeddings.npy')
ids = np.load('/kaggle/working/user_voice_ids.npy', allow_pickle=True)

print('=== Shape verification ===')
print(f'user_voice_embeddings.npy : {emb.shape}  dtype={emb.dtype}')
print(f'user_voice_ids.npy        : {ids.shape}  dtype={ids.dtype}')

print('\n=== Norm verification ===')
norms = np.linalg.norm(emb, axis=1)
print(f'Norm min={norms.min():.6f}  max={norms.max():.6f}  mean={norms.mean():.6f}')

print('\n=== Checks ===')
assert emb.shape[0] == ids.shape[0], 'Count mismatch!'
assert emb.shape[1] == 1024, f'Wrong dim: {emb.shape[1]}'
assert np.allclose(norms, 1.0, atol=1e-4), 'Not normalized!'
print(f'Users: {emb.shape[0]:,}')
print(f'Dim: {emb.shape[1]}')
print('ALL CHECKS PASSED! ✅')
print('\nDownload both files from Output tab on the right →')

=== Shape verification ===
user_voice_embeddings.npy : (94762, 1024)  dtype=float32
user_voice_ids.npy        : (94762,)  dtype=object

=== Norm verification ===
Norm min=1.000000  max=1.000000  mean=1.000000

=== Checks ===
Users: 94,762
Dim: 1024
ALL CHECKS PASSED! ✅

Download both files from Output tab on the right →
